# PB — Valor de negocio (gain curve, lift, umbral EV)

Notebook autocontenido: la lógica de valor de negocio está en celdas de este `.ipynb` (no en `src/`).

Replica el **Soft Voting** de `notebooks/14_final_validation.ipynb` para obtener `y_test` y `y_proba` con el mismo split (`random_state=42`, `test_size=0.2`, estratificado).

**Datos:** `../data/processed/04_default_credit_features.csv` (desde la carpeta `notebooks/`).


## 1. Sprint 1 (`01_business.ipynb`) vs Sprint 4

| Criterio | Sprint 1 | Sprint 4 (test, mismo modelo) |
|----------|----------|--------------------------------|
| ROC-AUC | ≥ 0.70 | Ver salida al entrenar |
| Recall | Alto (menos FN) | Depende del umbral |
| F1 | Complementario | ~ típico con umbral 0.5 |

## 2. Matriz costo–beneficio (positivo = intervenir)

| Resultado | Parámetro |
|-----------|-----------|
| TP | Beneficio `b` |
| FP | Costo `c_FP` |
| FN | Costo `c_FN` |

**Valor neto:** `b·TP − c_FP·FP − c_FN·FN`

Montos en código: **ilustrativos** (NT$); sustituir por datos del negocio.


In [ ]:
import numpy as np
import pandas as pd


def gain_lift_table(y_true: np.ndarray, y_score: np.ndarray) -> pd.DataFrame:
    # Gain acumulado y lift vs. orden aleatorio (score mora descendente).
    y_true = np.asarray(y_true).astype(int)
    y_score = np.asarray(y_score, dtype=float)
    order = np.argsort(-y_score)
    y_sorted = y_true[order]
    n = len(y_true)
    pos_total = int(y_true.sum())
    if pos_total == 0:
        raise ValueError("y_true no contiene positivos (mora).")
    ranks = np.arange(1, n + 1)
    pct_population = ranks / n
    cum_pos = np.cumsum(y_sorted)
    pct_positives_captured = cum_pos / pos_total
    lift = pct_positives_captured / np.maximum(pct_population, 1e-12)
    return pd.DataFrame(
        {
            "rank": ranks,
            "pct_population": pct_population,
            "pct_positives_captured": pct_positives_captured,
            "lift_vs_random": lift,
        }
    )


def confusion_at_threshold(y_true: np.ndarray, y_score: np.ndarray, threshold: float):
    y_true = np.asarray(y_true).astype(int)
    y_pred = (np.asarray(y_score) >= threshold).astype(int)
    tn = int(np.sum((y_true == 0) & (y_pred == 0)))
    fp = int(np.sum((y_true == 0) & (y_pred == 1)))
    fn = int(np.sum((y_true == 1) & (y_pred == 0)))
    tp = int(np.sum((y_true == 1) & (y_pred == 1)))
    return tn, fp, fn, tp


def expected_net_value(
    tp: int, fp: int, fn: int, benefit_tp: float, cost_fp: float, cost_fn: float
) -> float:
    return tp * benefit_tp - fp * cost_fp - fn * cost_fn


def threshold_search_ev(
    y_true: np.ndarray,
    y_score: np.ndarray,
    benefit_tp: float,
    cost_fp: float,
    cost_fn: float,
    thresholds: np.ndarray | None = None,
) -> pd.DataFrame:
    y_score = np.asarray(y_score, dtype=float)
    if thresholds is None:
        thresholds = np.unique(np.concatenate([[0.0], np.sort(y_score), [1.0]]))
    rows = []
    for t in thresholds:
        tn, fp, fn, tp = confusion_at_threshold(y_true, y_score, t)
        ev = expected_net_value(tp, fp, fn, benefit_tp, cost_fp, cost_fn)
        rows.append(
            {
                "threshold": t,
                "tp": tp,
                "fp": fp,
                "fn": fn,
                "tn": tn,
                "net_value": ev,
            }
        )
    return pd.DataFrame(rows)


def best_threshold_ev(
    y_true: np.ndarray,
    y_score: np.ndarray,
    benefit_tp: float,
    cost_fp: float,
    cost_fn: float,
):
    df = threshold_search_ev(y_true, y_score, benefit_tp, cost_fp, cost_fn)
    best_idx = int(df["net_value"].idxmax())
    return float(df.loc[best_idx, "threshold"]), df


In [ ]:
import os
import joblib
import matplotlib.pyplot as plt

import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.svm import SVC
from sklearn.ensemble import VotingClassifier, GradientBoostingClassifier, AdaBoostClassifier

from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import RandomOverSampler

TARGET = "default payment next month"

df = pd.read_csv("../data/processed/04_default_credit_features.csv")
df = df.drop(columns=["ID"])
X = df.drop(columns=[TARGET])
y = df[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

cat_cols = X_train.select_dtypes(
    include=["object", "category", "string"]
).columns.tolist()
num_cols = X_train.select_dtypes(include=["int64", "float64"]).columns.tolist()

numeric_pipeline = Pipeline(
    [
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]
)
categorical_pipeline = Pipeline(
    [
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OneHotEncoder(handle_unknown="ignore")),
    ]
)
preproc = ColumnTransformer(
    [
        ("num", numeric_pipeline, num_cols),
        ("cat", categorical_pipeline, cat_cols),
    ]
)

svm_best_params = {"C": 0.515, "kernel": "rbf", "gamma": "scale"}
gb_best_params = {"n_estimators": 69, "learning_rate": 0.087, "max_depth": 3}
ada_best_params = {"n_estimators": 81, "learning_rate": 0.09}

best_svm_soft = SVC(**svm_best_params, probability=True, random_state=42)
best_gb = GradientBoostingClassifier(**gb_best_params, random_state=42)
best_ada = AdaBoostClassifier(**ada_best_params, random_state=42)

voting_soft = VotingClassifier(
    estimators=[
        ("svm", best_svm_soft),
        ("gb", best_gb),
        ("ada", best_ada),
    ],
    voting="soft",
)

best_model = ImbPipeline(
    [
        ("preprocessing", preproc),
        ("sampling", RandomOverSampler(random_state=42)),
        ("model", voting_soft),
    ]
)

best_model.fit(X_train, y_train)
y_proba = best_model.predict_proba(X_test)[:, 1]

os.makedirs("../models", exist_ok=True)
joblib.dump(best_model, "../models/final_model.pkl")
print("Modelo entrenado; y_proba lista para valor de negocio.")


In [ ]:
# --- Supuestos económicos ilustrativos (NT$) ---
BENEFIT_TP = 18_000.0
COST_FP = 800.0
COST_FN = 22_000.0

N_TEST = len(y_test)
N_ANNUAL = 600_000

gl = gain_lift_table(y_test.values, y_proba)

k20 = max(1, int(0.20 * N_TEST)) - 1
row20 = gl.iloc[k20]
print("=== Gain / lift (top 20 % ordenados por score) ===")
print(f"Fracción población: {row20['pct_population']:.3f}")
print(f"Fracción morosos capturados: {row20['pct_positives_captured']:.3f}")
print(f"Lift vs. azar: {row20['lift_vs_random']:.3f}x")

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].plot(gl["pct_population"], gl["pct_positives_captured"], label="Modelo")
ax[0].plot([0, 1], [0, 1], "k--", alpha=0.5, label="Baseline aleatorio")
ax[0].set_xlabel("Fracción población (score desc.)")
ax[0].set_ylabel("Fracción morosos capturados")
ax[0].set_title("Gain curve")
ax[0].legend()
ax[0].grid(True, alpha=0.3)

ax[1].plot(gl["pct_population"], gl["lift_vs_random"])
ax[1].axhline(1.0, color="k", linestyle="--", alpha=0.5)
ax[1].set_xlabel("Fracción población")
ax[1].set_ylabel("Lift vs. azar")
ax[1].set_title("Lift acumulado")
ax[1].grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

best_t, ev_df = best_threshold_ev(y_test.values, y_proba, BENEFIT_TP, COST_FP, COST_FN)
best_row = ev_df.loc[ev_df["net_value"].idxmax()]
print("\n=== Umbral óptimo EV ===")
print(f"Umbral*: {best_t:.4f}")
print(best_row.astype(object))

tn0, fp0, fn0, tp0 = confusion_at_threshold(y_test.values, y_proba, 0.5)
ev_default = expected_net_value(tp0, fp0, fn0, BENEFIT_TP, COST_FP, COST_FN)
net_best = float(best_row["net_value"])
scale = N_ANNUAL / N_TEST

print("\n=== Impacto anual estimado (extrapolación lineal) ===")
print(f"Net test (umbral*): {net_best:,.0f} NT$")
print(f"Net anual (~{N_ANNUAL:,} cuentas): {net_best * scale:,.0f} NT$")
print(f"Net test (umbral 0.5): {ev_default:,.0f} NT$")

print("\n=== Sensibilidad ±30 % ===")
for f in (0.7, 1.0, 1.3):
    b, cf, cn = BENEFIT_TP * f, COST_FP * f, COST_FN * f
    t_star, _ = best_threshold_ev(y_test.values, y_proba, b, cf, cn)
    _, fp, fn, tp = confusion_at_threshold(y_test.values, y_proba, t_star)
    nv = expected_net_value(tp, fp, fn, b, cf, cn)
    print(f"x{f:.1f}: umbral*={t_star:.4f}, net_anual≈{nv * scale:,.0f} NT$")

print("\n=== Supuestos ===")
print("- N_ANNUAL y montos son hipótesis; calibrar con negocio.")
print("- Extrapolación lineal asume perfil similar al holdout.")
